Port-forward MLflow  
`kubectl port-forward -n mldata svc/mlflow-mlflow 5000:5000 &` 

Port-forward MLflow S3  
`kubectl port-forward svc/mlminio -n mldata 9000:9000 &`

In [6]:
import os
import sys
import io
import json
import boto3
import mlflow
import polars as pl
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
load_dotenv()
# MinIO (Data Storage)

MINIO_ENDPOINT = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
MINIO_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
MINIO_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
DATA_BUCKET = "processed-features"
TEST_DATA_KEY = "test_data.parquet" 

In [8]:
# MLflow (Model Registry)
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
EXPERIMENT_NAME = "mobile_sales_prediction"
MODEL_NAME = "mobile-sales-predictor"
ALIAS_CHAMPION = "champion"
TARGET_COLUMN  = "Quantity Sold"
R2_THRESHOLD = 0.01

In [9]:
s3 = boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY
    )

In [10]:
obj = s3.get_object(Bucket=DATA_BUCKET, Key=TEST_DATA_KEY)
data = io.BytesIO(obj['Body'].read())
df = pl.read_parquet(data)

In [11]:
df.columns

['Price',
 'days_to_sell',
 'dispatch_year',
 'dispatch_month',
 'dispatch_day_of_week',
 'spec_length',
 'brand_code',
 'region_code',
 'ram_code',
 'rom_code',
 'avg_price_per_brand',
 'avg_qty_per_region',
 'Quantity Sold']

In [12]:
df.shape

(4997, 13)

In [13]:
X_test = df.drop(TARGET_COLUMN).to_pandas()
y_test = df[TARGET_COLUMN].to_pandas()
X_test.columns

Index(['Price', 'days_to_sell', 'dispatch_year', 'dispatch_month',
       'dispatch_day_of_week', 'spec_length', 'brand_code', 'region_code',
       'ram_code', 'rom_code', 'avg_price_per_brand', 'avg_qty_per_region'],
      dtype='object')

In [22]:
def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    mae = mean_absolute_error(y, y_pred)
    rmse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}

In [15]:
mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.tracking.MlflowClient()

In [16]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

In [17]:
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
versions

[<ModelVersion: aliases=[], creation_timestamp=1789059845106, current_stage='None', description='', last_updated_timestamp=1789059845106, name='mobile-sales-predictor', run_id='d33f482e727c47e280b3f0afc4d00d52', run_link='', source='mlflow-artifacts:/1/d33f482e727c47e280b3f0afc4d00d52/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='9'>,
 <ModelVersion: aliases=[], creation_timestamp=1789059683996, current_stage='None', description='', last_updated_timestamp=1789059683996, name='mobile-sales-predictor', run_id='1293fcc798d44ae7bf198f0a43b35c50', run_link='', source='mlflow-artifacts:/1/1293fcc798d44ae7bf198f0a43b35c50/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='8'>,
 <ModelVersion: aliases=[], creation_timestamp=1789059417511, current_stage='None', description='', last_updated_timestamp=1789059417511, name='mobile-sales-predictor', run_id='8e1e50ec594841e2a52d5a5725baad00', run_link='', source='mlflow-artifacts:/1

In [18]:
latest_version = max(versions, key=lambda v: int(v.version))
latest_version


<ModelVersion: aliases=[], creation_timestamp=1789059845106, current_stage='None', description='', last_updated_timestamp=1789059845106, name='mobile-sales-predictor', run_id='d33f482e727c47e280b3f0afc4d00d52', run_link='', source='mlflow-artifacts:/1/d33f482e727c47e280b3f0afc4d00d52/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='9'>

In [19]:
RUN_ID = latest_version.run_id
RUN_ID

'd33f482e727c47e280b3f0afc4d00d52'

In [20]:
new_model = mlflow.sklearn.load_model(f"runs:/{RUN_ID}/model")

In [23]:
new_metrics = evaluate_model(new_model, X_test, y_test)

In [24]:
new_metrics

{'mae': 2.49207108670577,
 'rmse': 8.225910123416272,
 'r2': -7.37432053536935e-05}

In [25]:
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("test_mae", new_metrics["mae"])
    mlflow.log_metric("test_rmse", new_metrics["rmse"])
    mlflow.log_metric("test_r2", new_metrics["r2"])

In [26]:
champion_version = client.get_model_version_by_alias(MODEL_NAME, ALIAS_CHAMPION)
champion_version

<ModelVersion: aliases=['champion'], creation_timestamp=1788885169475, current_stage='None', description='', last_updated_timestamp=1788885169475, name='mobile-sales-predictor', run_id='757d23c7fadf454da35e7fc8f9640025', run_link='', source='mlflow-artifacts:/1/757d23c7fadf454da35e7fc8f9640025/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='1'>

In [27]:
champion_model = mlflow.sklearn.load_model(f"runs:/{champion_version.run_id}/model")

/mnt/e/Repos/Projects/ML-Model-Factory/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.4.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/e/Repos/Projects/ML-Model-Factory/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.4.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [28]:
champion_metrics = evaluate_model(champion_model, X_test, y_test)
champion_metrics

{'mae': 2.5073764258555133,
 'rmse': 8.449367780668402,
 'r2': -0.02724084476410793}

In [29]:
r2_improvement = new_metrics['r2'] - champion_metrics['r2']

In [30]:
r2_improvement

0.027167101558754236

In [31]:
if r2_improvement > R2_THRESHOLD:
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    target_version = None
    for v in versions:
        if v.run_id == RUN_ID:
            target_version = v.version
            break

    if target_version:
        client.set_registered_model_alias(MODEL_NAME, ALIAS_CHAMPION, target_version)